# Top 10 榜单策略：一手组合回测

本 Notebook 使用本地 WT8 CSV，分别回测 Long & Hold、小时 SAR 多头、小时 MACD 多头榜单。组合初始资金为 500,000 元；每个品种最多持有一手，组合最多同时持有 10 个品种。每个观察时点用回测起点至当时的 WT8 单品种权益变化排序，进入 Top 10 后在下一条可交易记录开仓，跌出 Top 10 后在下一条可交易记录平仓。

## 收益与杠杆口径

- 排名信号：`WT8权益(当时) - WT8权益(回测起点)`，与网站按起止时点计算的单品种策略收益金额一致。排行榜信号本身不乘杠杆。
- 一手名义价值：`价格 × 合约乘数`。合约乘数从成交记录的 `成交额 ÷ (价格 × 成交手数)` 反推。
- 一手价格盈亏：`(当前价格 - 上次价格) × 合约乘数`。期货盈亏已经是杠杆后的合约盈亏，不能再乘一次杠杆。
- 保证金收益率：`净盈亏 ÷ 入场保证金`。保证金率根据 WT8 持仓记录反推：`(权益 - 可用资金) ÷ (价格 × 合约乘数)`；按历史时点使用当时已观测到的比率。隐含杠杆约为 `1 ÷ 保证金率`。
- 费用：CSV 中手续费均为 0，因此基准为 0。可在参数格设置单边费率（基于名义成交额）；滑点、利息和涨跌停未建模。
- 连续合约换月：主连合约代码变化时，换月当根不把新旧合约价差记作盈亏，视作在该根价格无价差换月；缺少旧、新合约的同时行情，真实换月成本无法从现有 CSV 还原。

这是一个基于 WT8 导出数据和明确撮合假设的研究回测。实际收益会受成交时点、手续费、滑点、保证金调整和连续合约换月影响。


In [1]:
from collections import Counter, defaultdict
from dataclasses import dataclass
from datetime import datetime, timedelta
from html import escape
from itertools import groupby
from heapq import merge
from pathlib import Path
from statistics import median
from typing import NamedTuple
import csv
import math
import re
import statistics

# Notebook 可从项目根目录打开，也可从 notebooks/ 目录打开。
BASE_DIR = next((p for p in (Path.cwd(), *Path.cwd().parents) if (p / "data").is_dir()), None)
if BASE_DIR is None:
    raise FileNotFoundError("找不到项目 data/ 目录；请从 Futures_Ranking 或其 notebooks/ 目录运行。")

STRATEGY_DIRS = {
    "long_hold": "小时Buy_And_Hold",
    "sar_long": "小时SAR多头",
    "macd_long": "小时MACD多头",
}
STRATEGY_NAMES = {
    "long_hold": "Long & Hold（小时涨幅榜）",
    "sar_long": "小时 SAR 多头",
    "macd_long": "小时 MACD 多头",
}

INITIAL_CAPITAL = 500_000.0
TOP_K = 10
START_AT = "2024-01-02 09:30:00"
END_AT = None  # None 表示取三套数据共同覆盖到的最后时点
ONE_WAY_FEE_RATE = 0.0  # 单边手续费率；例如 0.0001 表示名义成交额的 1 bp

print("项目目录：", BASE_DIR)
print("回测区间：", START_AT, "至数据共同末时点")
print("初始资金：", f"{INITIAL_CAPITAL:,.0f}", "元；每品种 1 手；组合最多", TOP_K, "个品种")

项目目录： D:\HuaweiMoveData\Users\32874\Desktop\Futures Ranking\Futures_Ranking
回测区间： 2024-01-02 09:30:00 至数据共同末时点
初始资金： 500,000 元；每品种 1 手；组合最多 10 个品种


## 1. 读取和检查 WT8 明细

以下按 CSV 的固定列读取：时间、合约、信号、价格、平仓/开仓手数、成交额、手续费、可用资金、权益。读取时同时根据 BK/SP 还原 WT8 每个单品种策略自己的持仓状态，用于估算保证金率。


In [2]:
class Bar(NamedTuple):
    timestamp: datetime
    contract: str
    signal: str
    price: float | None
    turnover: float | None
    trade_qty: float | None
    equity: float
    available: float | None
    position: int
    fee: float | None


def parse_number(value):
    text = str(value or "").replace("\ufeff", "").strip().replace(",", "")
    if not text or text in {"---", "--", "-"}:
        return None
    try:
        return float(text)
    except ValueError:
        return None


def parse_timestamp(value):
    text = str(value or "").replace("\ufeff", "").strip()
    for fmt in ("%Y/%m/%d %H:%M:%S", "%Y-%m-%d %H:%M:%S", "%Y/%m/%d %H:%M", "%Y-%m-%d %H:%M"):
        try:
            return datetime.strptime(text, fmt)
        except ValueError:
            pass
    return None


def instrument_name(path):
    name = re.sub(r"_小时.*?_信号明细$", "", path.stem)
    name = re.sub(r"_信号明细$", "", name)
    return re.sub(r"\s+", "", name)


def read_bars(path):
    raw_rows = None
    last_error = None
    for encoding in ("gb18030", "gbk", "utf-8-sig"):
        try:
            with path.open("r", encoding=encoding, errors="strict", newline="") as handle:
                reader = csv.reader(handle)
                next(reader, None)
                raw_rows = list(reader)
            break
        except UnicodeDecodeError as exc:
            last_error = exc
    if raw_rows is None:
        raise UnicodeError(f"无法解码 {path.name}: {last_error}")

    bars = []
    position = 0
    for raw in raw_rows:
        if len(raw) < 15:
            continue
        timestamp = parse_timestamp(raw[1])
        equity = parse_number(raw[14])
        if timestamp is None or equity is None:
            continue  # 汇总行或无效行
        signal = str(raw[3] or "").strip()
        if signal == "BK":
            position = 1
            trade_qty = parse_number(raw[9])
        elif signal in {"SP", "SK", "BP"}:
            trade_qty = parse_number(raw[8])
            if signal == "SP":
                position = 0
            else:
                position = 0
        else:
            trade_qty = None
        bars.append(Bar(
            timestamp=timestamp,
            contract=str(raw[2] or "").strip(),
            signal=signal,
            price=parse_number(raw[7]),
            turnover=parse_number(raw[10]),
            trade_qty=trade_qty,
            equity=equity,
            available=parse_number(raw[13]),
            position=position,
            fee=parse_number(raw[11]),
        ))
    bars.sort(key=lambda bar: bar.timestamp)
    return bars


def format_dt(value):
    return value.strftime("%Y-%m-%d %H:%M:%S") if value else ""


def load_strategy_rows(strategy_id):
    directory = BASE_DIR / "data" / STRATEGY_DIRS[strategy_id]
    result = {}
    for path in sorted(directory.glob("*.csv")):
        name = instrument_name(path)
        bars = read_bars(path)
        if bars:
            result[name] = bars
    return result


print("明细读取函数已准备好。")

明细读取函数已准备好。


## 2. 从 WT8 交易和资金字段估算合约乘数、保证金率

乘数用三套 CSV 的 BK/SP 实际成交金额交叉核对。保证金率不是从网上套一个统一值，而是从每个品种持仓期间的 `权益 - 可用资金` 反推；同一时点若不同策略都有持仓，取这些观察的中位数。回测时只取不晚于当前时点的最近比率，不用未来保证金记录。


In [3]:
MULTIPLIER_SAMPLES = defaultdict(list)
RAW_MARGIN_SAMPLES = defaultdict(list)
DATA_AUDIT = {}
INSTRUMENTS_BY_STRATEGY = {}
DATA_FEES = Counter()

for strategy_id, folder in STRATEGY_DIRS.items():
    paths = sorted((BASE_DIR / "data" / folder).glob("*.csv"))
    row_count = 0
    signals = Counter()
    first_time = None
    last_time = None
    populated = set()
    for path in paths:
        name = instrument_name(path)
        bars = read_bars(path)
        if bars:
            populated.add(name)
        for bar in bars:
            row_count += 1
            first_time = bar.timestamp if first_time is None else min(first_time, bar.timestamp)
            last_time = bar.timestamp if last_time is None else max(last_time, bar.timestamp)
            if bar.signal:
                signals[bar.signal] += 1
            if bar.fee is not None:
                DATA_FEES[strategy_id] += bar.fee
            if bar.signal in {"BK", "SP"} and bar.price and bar.turnover and bar.trade_qty:
                MULTIPLIER_SAMPLES[name].append(bar.turnover / (bar.price * bar.trade_qty))
            if bar.position and bar.price and bar.equity is not None and bar.available is not None:
                margin_amount = bar.equity - bar.available
                if margin_amount > 0:
                    RAW_MARGIN_SAMPLES[name].append((bar.timestamp, margin_amount, bar.price))
    INSTRUMENTS_BY_STRATEGY[strategy_id] = populated
    DATA_AUDIT[strategy_id] = {
        "strategy": STRATEGY_NAMES[strategy_id],
        "files": len(paths),
        "with_rows": len(populated),
        "rows": row_count,
        "first": first_time,
        "last": last_time,
        "BK": signals["BK"],
        "SP": signals["SP"],
    }


CONTRACT_MULTIPLIER = {}
MULTIPLIER_DISPERSION = {}
for name, samples in MULTIPLIER_SAMPLES.items():
    center = median(samples)
    nearest_integer = round(center)
    multiplier = float(nearest_integer) if nearest_integer > 0 and abs(center - nearest_integer) <= max(0.01, center * 0.005) else center
    CONTRACT_MULTIPLIER[name] = multiplier
    MULTIPLIER_DISPERSION[name] = max(abs(value - multiplier) / multiplier for value in samples)


MARGIN_HISTORY = {}
MARGIN_STATS = {}
for name, samples in RAW_MARGIN_SAMPLES.items():
    multiplier = CONTRACT_MULTIPLIER.get(name)
    if not multiplier:
        continue
    dated_rates = []
    for timestamp, margin_amount, price in samples:
        notional = price * multiplier
        rate = margin_amount / notional if notional else None
        if rate is not None and 0.001 <= rate <= 0.50:
            dated_rates.append((timestamp, rate))
    dated_rates.sort(key=lambda item: item[0])
    timeline = []
    for timestamp, grouped in groupby(dated_rates, key=lambda item: item[0]):
        same_time_rates = [item[1] for item in grouped]
        timeline.append((timestamp, median(same_time_rates)))
    if timeline:
        MARGIN_HISTORY[name] = timeline
        all_rates = [item[1] for item in timeline]
        MARGIN_STATS[name] = {
            "samples": len(dated_rates),
            "observations": len(timeline),
            "median": median(all_rates),
            "min": min(all_rates),
            "max": max(all_rates),
        }
MARGIN_TIMES = {name: [item[0] for item in history] for name, history in MARGIN_HISTORY.items()}


COMMON_UNIVERSE = set.intersection(*(set(names) for names in INSTRUMENTS_BY_STRATEGY.values()))
COMMON_UNIVERSE &= set(CONTRACT_MULTIPLIER)
COMMON_UNIVERSE &= set(MARGIN_HISTORY)
ALL_SEEN_INSTRUMENTS = set.union(*(set(names) for names in INSTRUMENTS_BY_STRATEGY.values()))
EXCLUDED_INSTRUMENTS = sorted(ALL_SEEN_INSTRUMENTS - COMMON_UNIVERSE)
if not COMMON_UNIVERSE:
    raise RuntimeError("没有同时具备价格、合约乘数和保证金率的可回测品种。")

START_DT = parse_timestamp(START_AT)
latest_per_strategy = [DATA_AUDIT[key]["last"] for key in STRATEGY_DIRS]
COMMON_LAST_DT = min(latest_per_strategy)
END_DT = parse_timestamp(END_AT) if END_AT else COMMON_LAST_DT
if START_DT is None or END_DT is None or START_DT >= END_DT:
    raise ValueError("请设置有效回测区间，且起始时点必须早于结束时点。")

def margin_rate_at(name, timestamp):
    history = MARGIN_HISTORY.get(name, [])
    times = MARGIN_TIMES.get(name, [])
    idx = bisect_right(times, timestamp) - 1
    return history[idx][1] if idx >= 0 else None


from bisect import bisect_right

def margin_table_rows():
    rows = []
    for name in sorted(COMMON_UNIVERSE):
        stats = MARGIN_STATS[name]
        start_rate = margin_rate_at(name, START_DT)
        rows.append({
            "品种": name,
            "合约乘数": CONTRACT_MULTIPLIER[name],
            "乘数反推成交笔数": len(MULTIPLIER_SAMPLES[name]),
            "乘数最大偏差": MULTIPLIER_DISPERSION[name],
            "起始保证金率": start_rate,
            "起始隐含杠杆": 1 / start_rate if start_rate else None,
            "期间保证金率中位数": stats["median"],
            "期间保证金率最低": stats["min"],
            "期间保证金率最高": stats["max"],
            "WT8保证金观察数": stats["samples"],
        })
    return rows


def show_html(html):
    try:
        from IPython.display import HTML, display
        display(HTML(html))
    except ImportError:
        print("当前 Python 环境没有 IPython，Notebook 在 Jupyter 中会显示 HTML 图表和表格。")


def show_table(rows, columns, percent_columns=(), number_columns=()):
    head = "".join(f"<th>{escape(col)}</th>" for col in columns)
    body = []
    for row in rows:
        cells = []
        for col in columns:
            value = row.get(col)
            if value is None:
                rendered = "—"
            elif col in percent_columns:
                rendered = f"{value * 100:,.2f}%"
            elif col in number_columns:
                rendered = f"{value:,.2f}"
            elif isinstance(value, datetime):
                rendered = format_dt(value)
            elif isinstance(value, float):
                rendered = f"{value:,.4f}"
            else:
                rendered = str(value)
            cells.append(f"<td>{escape(rendered)}</td>")
        body.append("<tr>" + "".join(cells) + "</tr>")
    html = "<table style='border-collapse:collapse;font-size:13px'>"
    html += "<thead><tr>" + head + "</tr></thead><tbody>" + "".join(body) + "</tbody></table>"
    show_html(html)


audit_rows = []
for key in STRATEGY_DIRS:
    audit = DATA_AUDIT[key]
    audit_rows.append({
        "策略": audit["strategy"],
        "CSV文件": audit["files"],
        "有行情的品种": audit["with_rows"],
        "有效行数": audit["rows"],
        "BK开仓信号": audit["BK"],
        "SP平仓信号": audit["SP"],
        "开始": audit["first"],
        "结束": audit["last"],
        "CSV手续费合计": DATA_FEES[key],
    })
show_table(audit_rows, ["策略", "CSV文件", "有行情的品种", "有效行数", "BK开仓信号", "SP平仓信号", "开始", "结束", "CSV手续费合计"])
print(f"三套数据共同有完整参数的品种：{len(COMMON_UNIVERSE)} 个。未纳入：{', '.join(EXCLUDED_INSTRUMENTS) or '无'}（成交记录不足，无法由本地 WT8 明细验证一手乘数和保证金率）。")
print(f"有效区间：{format_dt(START_DT)} 至 {format_dt(END_DT)}；每个品种的榜单评分只使用不晚于当前时点的最近 WT8 记录。")
print("所有 CSV 的手续费字段合计为 0；基准回测也按 0 计费。")


三套数据共同有完整参数的品种：82 个。未纳入：强麦主连（成交记录不足，无法由本地 WT8 明细验证一手乘数和保证金率）。
有效区间：2024-01-02 09:30:00 至 2026-10-09 15:00:00；每个品种的榜单评分只使用不晚于当前时点的最近 WT8 记录。
所有 CSV 的手续费字段合计为 0；基准回测也按 0 计费。


策略,CSV文件,有行情的品种,有效行数,BK开仓信号,SP平仓信号,开始,结束,CSV手续费合计
Long & Hold（小时涨幅榜）,83,83,297211,1398,0,2024-01-02 02:00:00,2026-10-09 15:00:00,0.0000
小时 SAR 多头,83,83,296552,18291,17578,2024-01-02 02:00:00,2026-10-09 15:00:00,0.0000
小时 MACD 多头,83,83,296596,12220,11468,2024-01-02 02:00:00,2026-10-09 15:00:00,0.0000


## 3. 每个品种的一手合约参数

表内起始保证金率是回测起点之前/当时已经由 WT8 持仓记录观察到的比率；回测持仓保证金随着 WT8 后续已发生的保证金率更新。`保证金率期间中位数/最高/最低` 用来检视各品种、各时段的差异，不参与未来时点的参数回填。


In [4]:
margin_rows = margin_table_rows()
show_table(
    margin_rows,
    ["品种", "合约乘数", "乘数反推成交笔数", "乘数最大偏差", "起始保证金率", "起始隐含杠杆", "期间保证金率中位数", "期间保证金率最低", "期间保证金率最高", "WT8保证金观察数"],
    percent_columns={"乘数最大偏差", "起始保证金率", "期间保证金率中位数", "期间保证金率最低", "期间保证金率最高"},
    number_columns={"合约乘数", "乘数反推成交笔数", "起始隐含杠杆", "WT8保证金观察数"},
)
print("说明：隐含杠杆 = 1 ÷ 保证金率；例如 10% 保证金率约对应 10 倍名义敞口。实际可用杠杆受资金、风控和交易所/期货公司调整限制。")

说明：隐含杠杆 = 1 ÷ 保证金率；例如 10% 保证金率约对应 10 倍名义敞口。实际可用杠杆受资金、风控和交易所/期货公司调整限制。


品种,合约乘数,乘数反推成交笔数,乘数最大偏差,起始保证金率,起始隐含杠杆,期间保证金率中位数,期间保证金率最低,期间保证金率最高,WT8保证金观察数
1000股指主连,200.00,533.00,0.00%,12.00%,8.33,11.89%,8.44%,14.90%,"5,445.00"
20号胶主连,10.00,869.00,0.00%,9.00%,11.11,8.97%,7.85%,10.82%,"8,121.00"
300股指主连,300.00,566.00,0.00%,12.00%,8.33,11.95%,8.75%,13.69%,"5,445.00"
500股指主连,200.00,565.00,0.00%,12.00%,8.33,11.87%,8.61%,14.18%,"5,409.00"
50股指主连,300.00,588.00,0.00%,12.00%,8.33,11.94%,9.06%,13.32%,"5,428.00"
BR橡胶主连,5.00,869.00,0.00%,12.00%,8.33,11.95%,9.52%,15.04%,"8,109.00"
LU燃油主连,10.00,839.00,0.00%,16.00%,6.25,15.91%,8.91%,18.89%,"8,118.00"
PTA主连,5.00,804.00,0.00%,7.00%,14.29,7.00%,4.72%,8.69%,"8,014.00"
PVC主连,5.00,819.00,0.00%,7.00%,14.28,7.01%,5.49%,8.48%,"7,872.00"
三十年债主连,"10,000.00",648.00,0.00%,3.50%,28.57,3.49%,3.25%,3.68%,"6,977.00"


## 4. 无前视偏差的榜单持仓回测

每个新 WT8 观察时点先更新该时点所有品种的数据，再按累计 WT8 权益变化取 Top 10。排名变化只能在该品种的下一条行情记录上成交。若单笔新保证金会使可用资金为负、或组合已经持有 10 个品种，则该次开仓不成交，并在后续观察时点重新尝试。持仓期间若保证金需求超过权益，模型会在可成交的当前时点按榜单末位优先强制平仓；若品种停市且没有当前价格，资金缺口会保留到该品种下一条行情。旧、新主力合约代码变化时平滑处理换月价差。


In [5]:
def portfolio_margin(positions, timestamp):
    total = 0.0
    for name, position in positions.items():
        rate = margin_rate_at(name, timestamp)
        if rate is None:
            rate = position["last_margin_rate"]
        price = position["last_price"]
        if rate is not None and price is not None:
            total += price * CONTRACT_MULTIPLIER[name] * rate
    return total


def run_backtest(strategy_id):
    all_rows = load_strategy_rows(strategy_id)
    bars_by_instrument = {name: rows for name, rows in all_rows.items() if name in COMMON_UNIVERSE}
    instruments = sorted(bars_by_instrument)

    baseline_equity = {}
    latest = {}
    latest_index = {}
    for name, bars in bars_by_instrument.items():
        times = [bar.timestamp for bar in bars]
        idx = bisect_right(times, START_DT) - 1
        if idx >= 0:
            latest[name] = bars[idx]
            latest_index[name] = idx
            baseline_equity[name] = bars[idx].equity
        else:
            baseline_equity[name] = INITIAL_CAPITAL

    streams = []
    def bar_event_stream(bars, instrument, begin_index, stop_index):
        for row_index in range(begin_index, stop_index):
            yield (bars[row_index].timestamp, instrument, row_index)

    for name, bars in bars_by_instrument.items():
        times = [bar.timestamp for bar in bars]
        begin = bisect_right(times, START_DT)
        stop = bisect_right(times, END_DT)
        if begin < stop:
            streams.append(bar_event_stream(bars, name, begin, stop))
    event_stream = merge(*streams)

    positions = {}
    pending = {}
    trades = []
    rejected_entries = []
    gross_pnl = 0.0
    fees_paid = 0.0
    roll_count = 0
    equity_curve = [{
        "timestamp": START_DT,
        "equity": INITIAL_CAPITAL,
        "gross_pnl": 0.0,
        "fees": 0.0,
        "margin_used": 0.0,
        "available": INITIAL_CAPITAL,
        "positions": 0,
    }]

    def close_at(name, timestamp, price, reason):
        nonlocal fees_paid
        position = positions.pop(name)
        multiplier = CONTRACT_MULTIPLIER[name]
        exit_notional = price * multiplier
        exit_fee = exit_notional * ONE_WAY_FEE_RATE
        fees_paid += exit_fee
        net = position["gross_pnl"] - position["entry_fee"] - exit_fee
        trades.append({
            "strategy": STRATEGY_NAMES[strategy_id],
            "instrument": name,
            "entry_time": position["entry_time"],
            "exit_time": timestamp,
            "entry_contract": position["entry_contract"],
            "exit_contract": position["contract"],
            "entry_price": position["entry_price"],
            "exit_price": price,
            "multiplier": multiplier,
            "entry_notional": position["entry_notional"],
            "entry_margin": position["entry_margin"],
            "gross_pnl": position["gross_pnl"],
            "fees": position["entry_fee"] + exit_fee,
            "net_pnl": net,
            "margin_return": net / position["entry_margin"] if position["entry_margin"] else None,
            "holding_hours": (timestamp - position["entry_time"]).total_seconds() / 3600,
            "rolls": position["rolls"],
            "exit_reason": reason,
        })

    for timestamp, grouped_events in groupby(event_stream, key=lambda item: item[0]):
        rows_now = {}
        for _, name, index in grouped_events:
            bar = bars_by_instrument[name][index]
            rows_now[name] = (index, bar)

            if name in positions and bar.price is not None:
                position = positions[name]
                old_contract = position["contract"]
                if old_contract and bar.contract and old_contract != bar.contract:
                    # 连续主连切换时不将跨月价差计作价格盈亏。
                    position["rolls"] += 1
                    roll_count += 1
                elif position["last_price"] is not None:
                    change_pnl = (bar.price - position["last_price"]) * CONTRACT_MULTIPLIER[name]
                    position["gross_pnl"] += change_pnl
                    gross_pnl += change_pnl
                position["last_price"] = bar.price
                position["contract"] = bar.contract or old_contract
            latest[name] = bar
            latest_index[name] = index

        due = []
        for name, (index, bar) in rows_now.items():
            order = pending.get(name)
            if order and index >= order["due_index"]:
                due.append((name, index, bar, order))

        # 同一时点先平仓释放保证金，再按之前的名次顺序开仓。
        for name, index, bar, order in sorted((item for item in due if item[3]["action"] == "exit"), key=lambda item: item[0]):
            pending.pop(name, None)
            if name in positions and bar.price is not None:
                close_at(name, timestamp, bar.price, "跌出Top 10")

        due_entries = sorted((item for item in due if item[3]["action"] == "enter"), key=lambda item: (item[3]["rank"], item[0]))
        for name, index, bar, order in due_entries:
            pending.pop(name, None)
            if name in positions or bar.price is None:
                continue
            if len(positions) >= TOP_K:
                rejected_entries.append({"timestamp": timestamp, "instrument": name, "reason": "组合已有10个持仓"})
                continue
            rate = margin_rate_at(name, timestamp)
            if rate is None:
                rejected_entries.append({"timestamp": timestamp, "instrument": name, "reason": "该时点没有可用WT8保证金率"})
                continue
            multiplier = CONTRACT_MULTIPLIER[name]
            notional = bar.price * multiplier
            required_margin = notional * rate
            entry_fee = notional * ONE_WAY_FEE_RATE
            current_equity = INITIAL_CAPITAL + gross_pnl - fees_paid
            used_margin = portfolio_margin(positions, timestamp)
            if current_equity - used_margin - entry_fee < required_margin:
                rejected_entries.append({"timestamp": timestamp, "instrument": name, "reason": "可用资金不足"})
                continue
            fees_paid += entry_fee
            positions[name] = {
                "entry_time": timestamp,
                "entry_contract": bar.contract,
                "contract": bar.contract,
                "entry_price": bar.price,
                "last_price": bar.price,
                "entry_notional": notional,
                "entry_margin": required_margin,
                "entry_fee": entry_fee,
                "gross_pnl": 0.0,
                "last_margin_rate": rate,
                "rolls": 0,
            }

        # 当前已知数据的起点至时点权益变化组成榜单；同分按品种名称排序。
        scores = []
        for name, bar in latest.items():
            score = bar.equity - baseline_equity[name]
            scores.append((score, name))
        scores.sort(key=lambda item: (-item[0], item[1]))
        rank_by_name = {name: rank for rank, (_, name) in enumerate(scores, 1)}
        target_names = {name for _, name in scores[:TOP_K]}

        # 所有进出场决策均排到该品种的下一条行情记录成交。
        for name in set(positions) | set(pending) | target_names:
            desired = name in target_names
            held = name in positions
            existing = pending.get(name)
            wanted_action = "enter" if desired and not held else "exit" if held and not desired else None
            if existing and existing["action"] != wanted_action:
                pending.pop(name, None)
                existing = None
            if wanted_action and existing is None:
                current_idx = latest_index.get(name)
                if current_idx is None or current_idx + 1 >= len(bars_by_instrument[name]):
                    continue
                next_bar = bars_by_instrument[name][current_idx + 1]
                if next_bar.timestamp <= END_DT:
                    pending[name] = {
                        "action": wanted_action,
                        "due_index": current_idx + 1,
                        "rank": rank_by_name.get(name, TOP_K + 1),
                    }

        # 保证金不足时，仅用当前时点已有报价平掉榜单末位持仓；不使用未来价格。
        current_equity = INITIAL_CAPITAL + gross_pnl - fees_paid
        current_margin = portfolio_margin(positions, timestamp)
        candidates = sorted(
            (name for name in positions if name in rows_now and rows_now[name][1].price is not None),
            key=lambda name: (rank_by_name.get(name, TOP_K + 1), name),
            reverse=True,
        )
        for name in candidates:
            if current_margin <= current_equity:
                break
            pending.pop(name, None)
            close_at(name, timestamp, positions[name]["last_price"], "保证金不足强制平仓")
            current_equity = INITIAL_CAPITAL + gross_pnl - fees_paid
            current_margin = portfolio_margin(positions, timestamp)

        mark = portfolio_margin(positions, timestamp)
        equity = INITIAL_CAPITAL + gross_pnl - fees_paid
        equity_curve.append({
            "timestamp": timestamp,
            "equity": equity,
            "gross_pnl": gross_pnl,
            "fees": fees_paid,
            "margin_used": mark,
            "available": equity - mark,
            "positions": len(positions),
        })

    if len(equity_curve) == 1:
        raise RuntimeError(f"{STRATEGY_NAMES[strategy_id]} 在所选区间内没有行情事件。")

    final_timestamp = equity_curve[-1]["timestamp"]
    for name in sorted(list(positions)):
        position = positions[name]
        close_at(name, final_timestamp, position["last_price"], "样本结束强制平仓")
    final_equity = INITIAL_CAPITAL + gross_pnl - fees_paid
    equity_curve[-1].update({
        "equity": final_equity,
        "gross_pnl": gross_pnl,
        "fees": fees_paid,
        "margin_used": 0.0,
        "available": final_equity,
        "positions": 0,
    })

    peak = INITIAL_CAPITAL
    max_drawdown = 0.0
    for point in equity_curve:
        peak = max(peak, point["equity"])
        drawdown = point["equity"] / peak - 1 if peak else 0.0
        max_drawdown = min(max_drawdown, drawdown)

    elapsed_days = max((final_timestamp - START_DT).total_seconds() / 86400, 1 / 24)
    cagr = (final_equity / INITIAL_CAPITAL) ** (365.25 / elapsed_days) - 1 if final_equity > 0 else -1.0
    weighted_margin_seconds = 0.0
    elapsed_seconds = 0.0
    max_margin = 0.0
    max_positions = 0
    available_min = min(point["available"] for point in equity_curve)
    for current, following in zip(equity_curve, equity_curve[1:]):
        seconds = max((following["timestamp"] - current["timestamp"]).total_seconds(), 0.0)
        weighted_margin_seconds += current["margin_used"] * seconds
        elapsed_seconds += seconds
        max_margin = max(max_margin, current["margin_used"])
        max_positions = max(max_positions, current["positions"])
    average_margin = weighted_margin_seconds / elapsed_seconds if elapsed_seconds else 0.0
    total_net_pnl = final_equity - INITIAL_CAPITAL
    margin_roi = total_net_pnl / average_margin if average_margin else None
    wins = sum(1 for trade in trades if trade["net_pnl"] > 0)

    summary = {
        "策略": STRATEGY_NAMES[strategy_id],
        "品种数": len(bars_by_instrument),
        "回测区间": f"{format_dt(START_DT)} — {format_dt(final_timestamp)}",
        "平仓笔数": len(trades),
        "净盈亏": total_net_pnl,
        "期末权益": final_equity,
        "组合收益率": total_net_pnl / INITIAL_CAPITAL,
        "年化复合收益率": cagr,
        "最大回撤": max_drawdown,
        "胜率": wins / len(trades) if trades else 0.0,
        "时间加权平均保证金": average_margin,
        "最大保证金占用": max_margin,
        "累计收益/平均保证金": margin_roi,
        "按快照平均持仓数": sum(point["positions"] for point in equity_curve) / len(equity_curve),
        "最大持仓数": max_positions,
        "最低可用资金": available_min,
        "未成交开仓尝试": len(rejected_entries),
        "可用资金不足拒绝": sum(1 for item in rejected_entries if item["reason"] == "可用资金不足"),
        "组合持仓已满拒绝": sum(1 for item in rejected_entries if item["reason"] == "组合已有10个持仓"),
        "缺少保证金率拒绝": sum(1 for item in rejected_entries if item["reason"] == "该时点没有可用WT8保证金率"),
        "保证金强平笔数": sum(1 for trade in trades if trade["exit_reason"] == "保证金不足强制平仓"),
        "主连换月次数": roll_count,
        "手续费": fees_paid,
    }
    return {
        "summary": summary,
        "equity_curve": equity_curve,
        "trades": trades,
        "rejected_entries": rejected_entries,
    }


RESULTS = {}
for strategy_id in STRATEGY_DIRS:
    print("正在回测：", STRATEGY_NAMES[strategy_id])
    RESULTS[strategy_id] = run_backtest(strategy_id)
    curve = RESULTS[strategy_id]["equity_curve"]
    assert curve[0]["equity"] == INITIAL_CAPITAL
    assert max(point["positions"] for point in curve) <= TOP_K
    assert min(point["available"] for point in curve) >= -1e-6
    assert all(trade["entry_margin"] > 0 for trade in RESULTS[strategy_id]["trades"])
print("三个策略的回测已完成。")

正在回测： Long & Hold（小时涨幅榜）
正在回测： 小时 SAR 多头
正在回测： 小时 MACD 多头
三个策略的回测已完成。


## 5. 组合结果与权益曲线

“组合收益率”以 500,000 元初始资金为分母；“累计收益/平均保证金”以回测期间时间加权平均占用保证金为分母，直观看杠杆资金效率。单笔的保证金收益率也保存在交易明细中。最大回撤按整个小时级观察权益曲线计算。


In [6]:
summary_rows = [RESULTS[key]["summary"] for key in STRATEGY_DIRS]
show_table(
    summary_rows,
    ["策略", "品种数", "平仓笔数", "净盈亏", "期末权益", "组合收益率", "年化复合收益率", "最大回撤", "胜率", "时间加权平均保证金", "最大保证金占用", "累计收益/平均保证金", "按快照平均持仓数", "最大持仓数", "最低可用资金", "未成交开仓尝试", "可用资金不足拒绝", "组合持仓已满拒绝", "缺少保证金率拒绝", "保证金强平笔数", "主连换月次数", "手续费"],
    percent_columns={"组合收益率", "年化复合收益率", "最大回撤", "胜率", "累计收益/平均保证金"},
    number_columns={"净盈亏", "期末权益", "时间加权平均保证金", "最大保证金占用", "最低可用资金", "手续费"},
)


def equity_svg(results):
    width, height = 960, 440
    left, right, top, bottom = 78, 22, 38, 54
    plot_w, plot_h = width - left - right, height - top - bottom
    sampled = {}
    values = []
    for key, result in results.items():
        points = result["equity_curve"]
        step = max(1, len(points) // 700)
        selected = points[::step]
        if selected[-1] is not points[-1]:
            selected.append(points[-1])
        sampled[key] = selected
        values.extend(point["equity"] for point in selected)
    low, high = min(values), max(values)
    if high == low:
        high = low + 1
    pad = (high - low) * 0.06
    low -= pad
    high += pad
    colors = {"long_hold": "#187a63", "sar_long": "#3568c8", "macd_long": "#d17c20"}
    def x_at(index, count):
        return left + (plot_w * index / max(count - 1, 1))
    def y_at(value):
        return top + plot_h * (high - value) / (high - low)
    pieces = [f"<svg xmlns='http://www.w3.org/2000/svg' viewBox='0 0 {width} {height}' style='max-width:100%;height:auto;background:#fff;font-family:Arial,sans-serif'>"]
    for i in range(5):
        y = top + i * plot_h / 4
        val = high - i * (high - low) / 4
        pieces.append(f"<line x1='{left}' y1='{y:.1f}' x2='{width-right}' y2='{y:.1f}' stroke='#e7ebf0'/>")
        pieces.append(f"<text x='{left-10}' y='{y+4:.1f}' text-anchor='end' font-size='12' fill='#5c6573'>{val:,.0f}</text>")
    for key, selected in sampled.items():
        n = len(selected)
        path = " ".join(("M" if i == 0 else "L") + f" {x_at(i,n):.1f} {y_at(point['equity']):.1f}" for i, point in enumerate(selected))
        pieces.append(f"<path d='{path}' fill='none' stroke='{colors[key]}' stroke-width='2.2'/>")
    pieces.append(f"<text x='{left}' y='{height-17}' font-size='12' fill='#5c6573'>{escape(format_dt(START_DT))}</text>")
    pieces.append(f"<text x='{width-right}' y='{height-17}' text-anchor='end' font-size='12' fill='#5c6573'>{escape(format_dt(END_DT))}</text>")
    legend_x = left + 8
    for key in STRATEGY_DIRS:
        pieces.append(f"<line x1='{legend_x}' y1='20' x2='{legend_x+25}' y2='20' stroke='{colors[key]}' stroke-width='3'/>")
        pieces.append(f"<text x='{legend_x+32}' y='24' font-size='13' fill='#253044'>{escape(STRATEGY_NAMES[key])}</text>")
        legend_x += 260
    pieces.append("</svg>")
    return "".join(pieces)


show_html(equity_svg(RESULTS))
print("净值曲线是模拟组合净权益（含可配置手续费），不是 WT8 原策略自身的权益曲线。")

净值曲线是模拟组合净权益（含可配置手续费），不是 WT8 原策略自身的权益曲线。


策略,品种数,平仓笔数,净盈亏,期末权益,组合收益率,年化复合收益率,最大回撤,胜率,时间加权平均保证金,最大保证金占用,累计收益/平均保证金,按快照平均持仓数,最大持仓数,最低可用资金,未成交开仓尝试,可用资金不足拒绝,组合持仓已满拒绝,缺少保证金率拒绝,保证金强平笔数,主连换月次数,手续费
Long & Hold（小时涨幅榜）,82,648,"1,041,550.00","1,541,550.00",208.31%,50.18%,-66.82%,37.81%,"736,007.66","1,259,548.00",141.51%,9.4620,10,18.10,1757,1542,215,0,47,178,0.00
小时 SAR 多头,82,471,"2,176,055.00","2,676,055.00",435.21%,83.29%,-60.90%,43.10%,"767,198.28","1,239,445.00",283.64%,9.7260,10,80.30,836,694,142,0,52,175,0.00
小时 MACD 多头,82,404,"2,569,250.00","3,069,250.00",513.85%,92.59%,-62.21%,44.31%,"847,375.77","1,191,816.60",303.20%,9.7433,10,24.60,1123,1040,83,0,33,196,0.00


HTML table or chart

## 6. 交易明细抽样

下面分别列每个策略净盈亏最高和最低的若干笔。完整开平仓记录保存在 `RESULTS[策略ID]["trades"]` 中；如果希望导出全部记录，可运行最后一格。


In [7]:
trade_sample_rows = []
for strategy_id in STRATEGY_DIRS:
    trades = RESULTS[strategy_id]["trades"]
    extremes = sorted(trades, key=lambda trade: trade["net_pnl"])
    chosen = extremes[:5] + extremes[-5:]
    seen = set()
    for trade in chosen:
        key = (trade["entry_time"], trade["instrument"], trade["exit_time"])
        if key in seen:
            continue
        seen.add(key)
        trade_sample_rows.append({
            "策略": trade["strategy"],
            "品种": trade["instrument"],
            "开仓时间": trade["entry_time"],
            "平仓时间": trade["exit_time"],
            "开仓合约": trade["entry_contract"],
            "开仓价": trade["entry_price"],
            "平仓价": trade["exit_price"],
            "乘数": trade["multiplier"],
            "入场保证金": trade["entry_margin"],
            "净盈亏": trade["net_pnl"],
            "保证金收益率": trade["margin_return"],
            "持有小时": trade["holding_hours"],
            "换月次数": trade["rolls"],
            "平仓原因": trade["exit_reason"],
        })
show_table(
    trade_sample_rows,
    ["策略", "品种", "开仓时间", "平仓时间", "开仓合约", "开仓价", "平仓价", "乘数", "入场保证金", "净盈亏", "保证金收益率", "持有小时", "换月次数", "平仓原因"],
    percent_columns={"保证金收益率"},
    number_columns={"开仓价", "平仓价", "入场保证金", "净盈亏"},
)

策略,品种,开仓时间,平仓时间,开仓合约,开仓价,平仓价,乘数,入场保证金,净盈亏,保证金收益率,持有小时,换月次数,平仓原因
Long & Hold（小时涨幅榜）,原油主连,2026-03-06 22:00:00,2026-03-10 09:30:00,sc2604,755.30,650.50,"1,000.0000","147,026.00","-104,800.00",-71.28%,83.5000,0,跌出Top 10
Long & Hold（小时涨幅榜）,1000股指主连,2024-12-09 09:30:00,2025-01-02 13:00:00,IM2412,"6,333.20","5,783.80",200.0000,"149,265.60","-93,200.00",-62.44%,579.5000,1,保证金不足强制平仓
Long & Hold（小时涨幅榜）,1000股指主连,2024-10-08 09:30:00,2024-10-10 10:30:00,IM2410,"6,200.00","5,829.00",200.0000,"106,233.60","-74,200.00",-69.85%,49.0000,0,跌出Top 10
Long & Hold（小时涨幅榜）,500股指主连,2024-09-30 10:30:00,2024-10-11 14:00:00,IC2410,"5,788.00","5,485.00",200.0000,"107,467.20","-60,600.00",-56.39%,267.5000,0,跌出Top 10
Long & Hold（小时涨幅榜）,1000股指主连,2025-02-07 09:30:00,2025-04-07 09:30:00,IM2503,"6,094.00","5,532.40",200.0000,"142,862.40","-53,920.00",-37.74%,"1,416.0000",1,保证金不足强制平仓
Long & Hold（小时涨幅榜）,三十年债主连,2024-01-04 13:00:00,2025-12-26 09:30:00,TL2403,101.81,113.00,"10,000.0000","35,581.00","125,300.00",352.15%,"17,324.5000",8,跌出Top 10
Long & Hold（小时涨幅榜）,500股指主连,2025-08-15 13:00:00,2026-10-09 15:00:00,IC2509,"6,525.00","7,111.00",200.0000,"148,339.20","261,480.00",176.27%,"10,082.0000",5,样本结束强制平仓
Long & Hold（小时涨幅榜）,300股指主连,2024-09-25 10:30:00,2026-10-09 15:00:00,IF2410,"3,431.20","4,244.80",300.0000,"114,523.20","346,860.00",302.87%,"17,860.5000",9,样本结束强制平仓
Long & Hold（小时涨幅榜）,1000股指主连,2025-07-15 09:30:00,2026-10-09 15:00:00,IM2509,"6,272.60","7,015.80",200.0000,"139,176.00","363,720.00",261.34%,"10,829.5000",5,样本结束强制平仓
Long & Hold（小时涨幅榜）,沪金主连,2024-03-28 02:00:00,2026-10-09 15:00:00,au2406,517.78,908.64,"1,000.0000","80,692.80","374,740.00",464.40%,"22,213.0000",15,样本结束强制平仓


## 7. 导出完整结果（可选）

运行该单元会在项目根目录创建 `backtest_output/`，保存每个策略的组合净值曲线、完整交易明细、逐品种乘数/保证金估计，以及策略汇总表。回测参数或数据变化后，重新运行前面的单元格再导出。


In [8]:
def write_csv(path, rows, fieldnames):
    path.parent.mkdir(parents=True, exist_ok=True)
    with path.open("w", encoding="utf-8-sig", newline="") as handle:
        writer = csv.DictWriter(handle, fieldnames=fieldnames, extrasaction="ignore")
        writer.writeheader()
        for row in rows:
            output = dict(row)
            for key, value in output.items():
                if isinstance(value, datetime):
                    output[key] = format_dt(value)
            writer.writerow(output)


OUTPUT_DIR = BASE_DIR / "backtest_output"
write_csv(OUTPUT_DIR / "strategy_summary.csv", summary_rows, list(summary_rows[0]))
write_csv(OUTPUT_DIR / "margin_assumptions.csv", margin_rows, list(margin_rows[0]))
for strategy_id, result in RESULTS.items():
    write_csv(
        OUTPUT_DIR / f"trades_{strategy_id}.csv",
        result["trades"],
        list(result["trades"][0]) if result["trades"] else ["strategy", "instrument", "entry_time", "exit_time", "net_pnl"],
    )
    write_csv(
        OUTPUT_DIR / f"equity_curve_{strategy_id}.csv",
        result["equity_curve"],
        list(result["equity_curve"][0]),
    )
print("已导出到：", OUTPUT_DIR)

已导出到： D:\HuaweiMoveData\Users\32874\Desktop\Futures Ranking\Futures_Ranking\backtest_output
